# **CardiAg: Behavioral Analysis from Intentional Binding task**

Pipeline for the behavioral analysis and overview of results from the Intentional Binding (IB task; Haggard et al., 2002), as part of the CardiAg study. 

Author: Marta Gerosa

Created on: 16 December 2024

Last update: 22 April 2026

## **Pipeline structure**

* **1. Data import and formatting**: load preprocessed behavioral data from `derivatives\beh-preproc` and concatenate into long format. 
* **2. Summary of mJE per subject**: calculate the mean judgment error (mJE) for action (`JE_act`) and tone (`JE_tone`) trials per subject, and extract group-level behavioral features. 
* **3. Computation of Action Binding & Tone Binding**: compute group-level action binding (mJE of OpA - BasA) and tone binding (mJE of OpT - BasT). This corresponds to Results section 2.1. 
* **4. Plot mJE distributions for action and tone trials (Figure 2a)**: plot group-level distributions of mJE, separately for action (BasA, OpA) and tone (BasT, OpT) conditions. This corresponds to Figure 2a.
* **5. Plot arrow summary of action and tone binding (Figure 2b)**: plot group-level overview of action binding and tone binding effects at the group level, including mJE for the Baseline (BasA, BasT) and Operant conditions (OpA, OpT), as well arrows scaled proportionally to direction and magnitude of effect. This corresponds to Figure 2b. 

In [1]:
############## Import modules ##############

import pandas as pd
import os
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import ptitprince as pt
import seaborn as sns
import re

## **1. Data import and formatting**

- Specify BIDS directory of `derivatives\beh-preproc` and load preprocessed behavioral data for participants (sub-16 & sub-33 excluded due to extreme JE distribution, sub-14 missing HW triggers)
- Concatenate all participant data into a long-formatted df (`behpreproc_long`)

In [ ]:
############## Load preprocessed behavioral data ##############

# sub-16 excluded due to extreme JE distribution, sub-14 due to missing HW triggers
# sub-33 excluded due to outlier JE distribution
participant_ids = [1, 2, 3, 5, 6, 12, 13, 15, 17, 18, 19,       
                   20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30,  
                   31, 32, 34, 35, 36, 37, 38, 39, 41, 42,      
                   43, 44, 45, 46, 47, 48, 49, 51, 53, 54, 55, 57] 

# Specify the data path info
wd = r'.\data'                  # directory of data storage
exp_name = 'CardiAgIBTask' 
datatype_name = 'beh'           # current datatype folder according to BIDS

# Initialize empty list to store dfs with beh preprocessed data for all participants
subj_dfs = []

# Iterate through each participant
for subj in participant_ids:

    subj_id = 'sub-' + str(subj) # participant ID (in BIDS format)
    subj_behpreproc_fname = f'{subj_id}_task-{exp_name}_beh-preproc.tsv'

    # Merge information into complete datapath
    behpreproc_dir = os.path.join(wd, 'derivatives', 'beh-preproc', subj_id, 
                                  datatype_name, subj_behpreproc_fname)

    # Read TSV file into a dataframe
    subj_behpreproc = pd.read_csv(behpreproc_dir, sep='\t', header=0)

    # Reset the index and append df to list
    subj_behpreproc.reset_index(drop=True, inplace=True)
    subj_dfs.append(subj_behpreproc)

# Create a long data frame with beh preprocessed data from all participants
behpreproc_long = pd.concat(subj_dfs, ignore_index=True)
behpreproc_long.rename(columns={'Unnamed: 0': 'subj_index'}, inplace=True)

## **2. Summary of mJE per subject**

- Group the data by subject and condition, calculating the **mean Judgement Error (mJE) and its standard deviation (sdJE)**, separately for the action (`JE_act`) and tone (`JE_tone`) conditions.
- Transform the summary into wide format (`IBmeasures_summary`), keeping one row per subject and a column per condition. 
- Group-level summary of other behavioral features during the task (e.g., % excluded NoResponse trials, wait time to keypress) from the `_beh-preproc-summary.md` file

In [ ]:
############## Summary of mJE for each subject and condition ##############

# Summarize mJE (sd) for action and tone trials by condition and subject
IBmeasures_summary = behpreproc_long.groupby(["subjID", "condition"]).agg(
    mJE_act = ("JE_act", "mean"),
    sdJE_act = ("JE_act", "std"),
    mJE_tone = ("JE_tone","mean"),
    sdJE_tone = ("JE_tone","std"))

# Transform summary into wide format (one row per subject) and drop columns with NaN (e.g., BasT with no action report)
IBmeasures_summary = IBmeasures_summary.reset_index().pivot(index='subjID', columns='condition').dropna(axis=1)

# Show head of mJE (SD) summary per subject and condition
IBmeasures_summary.head()

In [5]:
############## Group-level summary of other behavioral features during task ##############

# Initialize empty list to store dfs with summary of behavioral task features for all participants
beh_summary_subj = []

# Iterate through each participant
for subj in participant_ids:

    subj_id = 'sub-' + str(subj) # participant ID (in BIDS format)
    subj_behsum_fname = f'{subj_id}_task-{exp_name}_beh-preproc-summary.md'

    # Merge information into complete datapath
    behsum_dir = os.path.join(wd, 'derivatives', 'beh-preproc', subj_id, 
                              datatype_name, subj_behsum_fname)

    # Import markdown file with summary of behavioral features
    with open(behsum_dir, 'r', encoding='utf-8') as f:
        behsum_text = f.read()

    # Use regex to match text regarding NoResponse trials
    noresp_match = re.search(r"did not initiate any keypress in (\d+)", behsum_text)
    noresp_trials = int(noresp_match.group(1)) if noresp_match else None

    # Use regex to match text regarding N of clock rotations & wait time to keypress
    wait_match = re.search(r"at ([\d.]+) clock rotations \(approx\. wait time to keypress of ([\d.]+) seconds\)", behsum_text)
    wait_rotations = float(wait_match.group(1))
    wait_time = float(wait_match.group(2))

    # Append participant data to list 
    beh_summary_subj.append({'subjID': subj_id, 
                             'noresponse_trials': noresp_trials, 
                             'noresponse_percent': round((noresp_trials/240)*100,2), 
                             'wait_rotations_keypress': wait_rotations,
                             'wait_time_keypress': wait_time})

beh_summary = pd.DataFrame(beh_summary_subj)

# Print group-level summary of behavioral features from intentional binding task
print(f"We excluded an average of {round(beh_summary['noresponse_trials'].mean(),2)} No Response trials per participant.")
print(f"This amounts to a total of {beh_summary['noresponse_trials'].sum()} No Response trials excluded from further analyses ({round(beh_summary['noresponse_percent'].mean(),2)}% of all trials).")
print(f"On average, participants waited {round(beh_summary['wait_rotations_keypress'].mean(),2)} clock rotations (i.e., {round(beh_summary['wait_time_keypress'].mean(),2)} seconds) before initiating the keypress in Action trials.")

We excluded an average of 3.91 No Response trials per participant.
This amounts to a total of 172 No Response trials excluded from further analyses (1.63% of all trials).
On average, participants waited 1.57 clock rotations (i.e., 4.03 seconds) before initiating the keypress in Action trials.


## **3. Computation of Action Binding & Tone Binding**

- Compute **action binding** at the group level, as the difference between the mJE of the OpA and BasA conditions.
- Compute **tone binding** at the group level, as the difference between the mJE of the OpT and BasT conditions.

Positive values indicate a delayed awareness of the event in the Operant condition (action+tone) compared to its respective Baseline condition (only action OR tone), while negative values indicate an anticipatory awareness of the event in the Operant compared to the Baseline condition. 

In [6]:
############## Action binding ##############

# Calculate overall mean and sd of mJE for BasA and OpA condition
mJE_act_BasA = IBmeasures_summary[('mJE_act', 'BasA')].mean()
sdJE_act_BasA = IBmeasures_summary[('mJE_act', 'BasA')].std()
mJE_act_OpA = IBmeasures_summary[('mJE_act', 'OpA')].mean()
sdJE_act_OpA = IBmeasures_summary[('mJE_act', 'OpA')].std()

# Calculate mean and sd of action binding (i.e., perceptual shifts between OpA-BasA) for each subject
IBmeasures_summary['act_binding_subj_mean'] = IBmeasures_summary[('mJE_act', 'OpA')] - IBmeasures_summary[('mJE_act', 'BasA')]
IBmeasures_summary['act_binding_subj_sd'] = IBmeasures_summary[('sdJE_act', 'OpA')] - IBmeasures_summary[('sdJE_act', 'BasA')]

# Calculate overall mean (sd) of action binding
act_binding_mean = IBmeasures_summary['act_binding_subj_mean'].mean()
act_binding_sd = IBmeasures_summary['act_binding_subj_sd'].mean()
act_binding_sd_avgje = (sdJE_act_BasA + sdJE_act_OpA) / 2

# Print summary of action binding
print("Action binding summary:")
print(f"mJE (SD) of trials in BasA condition: {mJE_act_BasA:.2f} ({sdJE_act_BasA:.2f}) ms")
print(f"mJE (SD) of trials in OpA condition: {mJE_act_OpA:.2f} ({sdJE_act_OpA:.2f}) ms")
print(f"Action binding mean (SD): {act_binding_mean:.2f} ({act_binding_sd_avgje:.2f}) ms")

Action binding summary:
mJE (SD) of trials in BasA condition: 26.97 (64.01) ms
mJE (SD) of trials in OpA condition: 59.84 (63.07) ms
Action binding mean (SD): 32.87 (63.54) ms


In [7]:
############## Tone binding ##############

# Calculate overall mean and sd of mJE for BasA and OpA condition
mJE_tone_BasT = IBmeasures_summary[('mJE_tone', 'BasT')].mean()
sdJE_tone_BasT = IBmeasures_summary[('mJE_tone', 'BasT')].std()
mJE_tone_OpT = IBmeasures_summary[('mJE_tone', 'OpT')].mean()
sdJE_tone_OpT = IBmeasures_summary[('mJE_tone', 'OpT')].std()

# Calculate mean and sd of tone binding (i.e., perceptual shifts between OpT-BasT) for each subject
IBmeasures_summary['tone_binding_subj_mean'] = IBmeasures_summary[('mJE_tone', 'OpT')] - IBmeasures_summary[('mJE_tone', 'BasT')]
IBmeasures_summary['tone_binding_subj_sd'] = IBmeasures_summary[('sdJE_tone', 'OpT')] - IBmeasures_summary[('sdJE_tone', 'BasT')]

# Calculate overall mean (sd) of tone binding
tone_binding_mean = IBmeasures_summary['tone_binding_subj_mean'].mean()
tone_binding_sd = IBmeasures_summary['tone_binding_subj_sd'].mean()
tone_binding_sd_avgje = (sdJE_tone_BasT + sdJE_tone_OpT) / 2

# Print summary of tone binding
print("Tone binding summary:")
print(f"mJE (SD) of trials in BasT condition: {mJE_tone_BasT:.2f} ({sdJE_tone_BasT:.2f}) ms")
print(f"mJE (SD) of trials in OpT condition: {mJE_tone_OpT:.2f} ({sdJE_tone_OpT:.2f}) ms")
print(f"Tone binding mean (SD): {tone_binding_mean:.2f} ({tone_binding_sd_avgje:.2f}) ms")

Tone binding summary:
mJE (SD) of trials in BasT condition: 42.68 (62.60) ms
mJE (SD) of trials in OpT condition: -45.50 (94.99) ms
Tone binding mean (SD): -88.18 (78.80) ms


## **4. Plot mJE distributions for action and tone trials (Figure 2a)**

- Plot **frequency distributions of mJE from individual subjects**, separately for Baseline and Operant conditions in action (BasA, OpA) and tone (BasT, OpT) trials. This is reported in Figure 2a. 

In [ ]:
# Create directory for saving plots
project_root = os.path.dirname(wd)
plotting_dir = os.path.join(project_root, 'results', 'datavisualization')
if not os.path.exists(plotting_dir):
    os.makedirs(plotting_dir)

In [ ]:
############## Plot JE distributions for action and tone trials (Figure 2a) ##############

### FIGURE 2a ###

colpalette = ['#007a4e','#a4cbb6','#0b3142','#7796cb','#b5a6b1']
fontsize = "x-large"; nbins = 15

# Function to compute mJE per pair of condition (e.g., action or tone) and plot their distributions
def plot_jedistr(ax, mJE_cond1, mJE_cond2, cond1_name, cond2_name, cond1_color, cond2_color, textpos1, textpos2, xlabel, ylabel):
    ax.hist(mJE_cond1, bins=nbins, alpha=1, label=cond1_name, color=cond1_color)
    ax.hist(mJE_cond2, bins=nbins, alpha=0.7, label=cond2_name, color=cond2_color)
    for avg_JE, label, color, x_offset in zip([mJE_cond1, mJE_cond2], 
                                                [cond1_name, cond2_name], 
                                                [cond1_color, cond2_color], 
                                                [textpos1, textpos2]):
        ax.axvline(avg_JE.mean(), color=color, linestyle='dashed', linewidth=1.2)
        ax.text(x=avg_JE.mean() * x_offset, y=15 * 0.75, fontsize=fontsize,
                s=f'{label} mean (SD):\n{avg_JE.mean():.2f} ({avg_JE.std():.2f}) ms', 
                color=color, ha='right' if x_offset < 0 else 'left')
    ax.set_xlim(-400, 400)
    ax.set_ylim(0, 16)
    ax.tick_params(axis='both', labelsize='large')
    ax.set_xlabel(xlabel, fontsize=fontsize)
    ax.set_ylabel(ylabel, fontsize=fontsize)
    ax.legend(fontsize='large')

# Compute the average JE per participant for each condition
avg_JE = {
    'action': {
        'BasA': behpreproc_long[behpreproc_long['condition'] == 'BasA'].groupby('subjID')['JE_act'].mean(),
        'OpA': behpreproc_long[behpreproc_long['condition'] == 'OpA'].groupby('subjID')['JE_act'].mean()},
    'tone': {
        'BasT': behpreproc_long[behpreproc_long['condition'] == 'BasT'].groupby('subjID')['JE_tone'].mean(),
        'OpT': behpreproc_long[behpreproc_long['condition'] == 'OpT'].groupby('subjID')['JE_tone'].mean()}
}

# Create subplots for distributions of action vs. tone trials
fig, axs = plt.subplots(1, 2, figsize=(12, 6), dpi=300)

# Plot mJE distribution for action trials (BasA, OpA)
plot_jedistr(axs[0], avg_JE['action']['BasA'], avg_JE['action']['OpA'], 'BasA', 'OpA', colpalette[1], colpalette[0], -0.4, 1.5,
        'Mean Judgment Error (mJE) for action trials (ms)', 'Frequency')

# Plot mJE distribution for tone trials (BasT, OpT)
plot_jedistr(axs[1], avg_JE['tone']['BasT'], avg_JE['tone']['OpT'], 'BasT', 'OpT', colpalette[3], colpalette[2], 1.6, 7.9,
        'Mean Judgment Error (mJE) for tone trials (ms)', 'Frequency')

plt.tight_layout()

# Save plot 
fig2_1 = os.path.join(plotting_dir, "Figure2a_mJE_distribution.svg")
plt.savefig(fig2_1, format="svg")
plt.show()

## **5. Plot arrow summary of action and tone binding (Figure 2b)**

- Plot **overview of action binding and tone binding effects at the group level**, reporting mJE for the Baseline (BasA, BasT) and Operant conditions (OpA, OpT), as well arrows scaled proportionally to direction and magnitude of effect. This is reported in Figure 2b. 

In [ ]:
############## Action & tone binding summary arrow plot (Figure 2b) ##############

### FIGURE 2b ###

# Set figure size and dpi
fig, ax = plt.subplots(figsize=(7.5, 3), dpi=300)

# Draw a horizontal line and labels for time
time_axis = patches.FancyArrowPatch((-50, 0.8), (320, 0.8), color='k', arrowstyle='-|>', mutation_scale=8)
ax.add_patch(time_axis)
ax.text(x=125, y=0.83, s='250 ms', color='k', ha='center', va='center', fontsize=9) # delay label
ax.text(x=310, y=0.76, s='Time', color='k', ha='right', va='center', fontsize=9) # time axis label
int_arrow = patches.FancyArrowPatch((0, 0.77), (250, 0.77), color='grey', arrowstyle='<|-|>', 
                                    mutation_scale=8, linewidth=0.8)
ax.add_patch(int_arrow)
ax.text(x=125, y=0.73, s='Actual interval', color='grey', ha='center', va='center', fontsize=9) # actual interval label

# Draw line for actual Action time (0 ms)
ax.axvline(x=0, color='k', linestyle='-', linewidth=4, ymin=0.75, ymax=0.85)
ax.text(x=0, y=0.92, s='Action', color='k', ha='center', va='center', fontsize=13)

# Draw line for actual Tone time (250ms)
ax.axvline(x=250, color='k', linestyle='-', linewidth=4, ymin=0.75, ymax=0.85)
ax.text(x=250, y=0.92, s='Tone', color='k', ha='center', va='center', fontsize=13)

# Draw perceived Baseline times
ax.axvline(x=mJE_act_BasA, color=colpalette[1], linestyle='-', linewidth=4, ymin=0.6, ymax=0.7)
ax.axvline(x=mJE_act_BasA, color=colpalette[1], linestyle='--', linewidth=0.5, ymin=0.5, ymax=0.65, alpha=0.6)
ax.axvline(x=(250 + mJE_tone_BasT), color=colpalette[3], linestyle='-', linewidth=4, ymin=0.6, ymax=0.7)
ax.axvline(x=(250 + mJE_tone_BasT), color=colpalette[3], linestyle='--', linewidth=0.5, ymin=0.5, ymax=0.65, alpha=0.6)
ax.text(x=-100, y=0.65, s='Baseline conditions', color='k', ha='left', va='center', fontsize=11)

# Draw perceived Operant times
ax.axvline(x=mJE_act_OpA, color=colpalette[0], linestyle='-', linewidth=4, ymin=0.45, ymax=0.55)
ax.axvline(x=mJE_act_OpA, color=colpalette[0], linestyle='--', linewidth=0.5, ymin=0.15, ymax=0.5, alpha=0.6)
ax.axvline(x=(250 + mJE_tone_OpT), color=colpalette[2], linestyle='-', linewidth=4, ymin=0.45, ymax=0.55)
ax.axvline(x=(250 + mJE_tone_OpT), color=colpalette[2], linestyle='--', linewidth=0.5, ymin=0.15, ymax=0.5, alpha=0.6)
ax.text(x=-100, y=0.5, s='Operant conditions', color='k', ha='left', va='center', fontsize=11)
ax.text(x=mJE_act_OpA, y=0.1, s='Perceived action', color=colpalette[0], ha='center', va='center', fontsize=9)
ax.text(x=(250 + mJE_tone_OpT), y=0.1, s='Perceived tone', color=colpalette[2], ha='center', va='center', fontsize=9)

# Draw arrow and add text for Action Binding
ax.arrow(x=(mJE_act_BasA+1), y=0.5, dx=(mJE_act_OpA - mJE_act_BasA)*0.9, dy=0, color=colpalette[0], 
         length_includes_head=True, head_width=0.015, head_length=2, linewidth=2.5)
ax.text(x=(mJE_act_BasA + (mJE_act_OpA - mJE_act_BasA)/2)*0.85, y=0.38, weight='bold',
        s='Action\n binding', color=colpalette[0], ha='center', va='center', fontsize=11)
ax.text(x=(mJE_act_BasA + (mJE_act_OpA - mJE_act_BasA)/2)*0.85, y=0.28, weight='bold',
        s=f'{act_binding_mean:.2f} ({act_binding_sd_avgje:.2f}) ms', 
        color=colpalette[0], ha='center', va='center', fontsize=9)

# Draw arrow and add text for Tone Binding
ax.arrow(x=(250 + mJE_tone_BasT), y=0.5, dx=(mJE_tone_OpT - mJE_tone_BasT)*0.93, dy=0, color=colpalette[2], 
         length_includes_head=True, head_width=0.015, head_length=2, head_starts_at_zero=True, linewidth=2.5)
ax.text(x=((250 + mJE_tone_BasT)+((mJE_tone_OpT - mJE_tone_BasT)/2)*0.85), y=0.38, weight='bold',
        s=f'Tone\n binding', color=colpalette[2], ha='center', va='center', fontsize=11)
ax.text(x=((250 + mJE_tone_BasT)+((mJE_tone_OpT - mJE_tone_BasT)/2)*0.85), y=0.28, weight='bold',
        s=f'{tone_binding_mean:.2f} ({tone_binding_sd_avgje:.2f}) ms', 
        color=colpalette[2], ha='center', va='center', fontsize=9)

# Add arrow for perceived interval
perc_arrow = patches.FancyArrowPatch((mJE_act_OpA, 0.5), (250+mJE_tone_OpT, 0.5), color='grey', 
                                     arrowstyle='<|-|>', mutation_scale=8, linewidth=0.8)
ax.add_patch(perc_arrow)
ax.text(x=125, y=0.46, s='Perceived interval', color='grey', ha='center', va='center', fontsize=9) # perceived interval label

# Set limits and remove axes
ax.set_xlim(-100, 350)
ax.set_ylim(0, 1)
ax.axis('off')

plt.tight_layout()

# Save plot 
fig2_2 = os.path.join(plotting_dir, "Figure2b_arrow_summary.svg")
plt.savefig(fig2_2, format="svg")
plt.show()